# Notebook 01 — Tracking Pipeline

This notebook walks through every stage of the tracking pipeline interactively.
It is equivalent to running `scripts/run_tracking.py` but lets you inspect
intermediate outputs at each step.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Hungarian stitching → stitched long CSV
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [1]:
import sys
sys.path.insert(0, '..')

import json
import os
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long, build_tracklets, stitch
from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
from src.visualization import render_vial_overlay_video

[03/30/26 15:59:47] WARNING  Your inference package version 1.1.1 is out of date! Please upgrade to  ]8;id=347776;file://c:\Users\Nabiya\anaconda3\envs\fly-tracking\Lib\site-packages\inference\core\__init__.py\__init__.py]8;;\:]8;id=701454;file://c:\Users\Nabiya\anaconda3\envs\fly-tracking\Lib\site-packages\inference\core\__init__.py#41\41]8;;\
                             version 1.2.0 of inference for the latest features and bug fixes by                   
                             running `pip install --upgrade inference`.                                            

ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.To suppress this warning, set CORE_MODEL_SAM_ENABLED to False.
ModelDependencyMissing: Your `inference` configuration does not support SAM3 model. Install SAM3 dependencies and set CORE_MODEL_SAM3_ENABLED to True.
ModelDependencyMissing: Your `inference` configuration does not support Gaze Detection model. Use pip install 'inference[gaze]' to install missing requirements.To suppress this warning, set CORE_MODEL_GAZE_ENABLED to False.
ModelDependencyMissing: Your `inference` configuration does not support YoloWorld model. Use pip install 'inference[yolo-world]' to install missing requirements.To suppress this warning, set CORE_MODEL_YOLO_WORLD_ENABLED to False.


## 1 — Configuration

Set your paths and Roboflow credentials here.

In [3]:
# ---- EDIT THESE ----
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/001/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted.mp4"
API_KEY   = "WktUdX9wCcMG4P7Iied4"
MODEL_ID  = "flies-123/1"   # e.g. "flies-123/1"

# Load defaults from config.yaml (override below if needed)
with open("../config.yaml") as _f:
    _cfg = yaml.safe_load(_f)
_t = _cfg.get("tracker", {})
_s = _cfg.get("stitching", {})
_p = _cfg.get("preprocessing", {})

confidence              = _t.get("confidence", 0.10)
lost_track_buffer       = _t.get("lost_track_buffer", 90)
min_matching_threshold  = _t.get("minimum_matching_threshold", 0.01)
min_consecutive_frames  = _t.get("minimum_consecutive_frames", 10)
asso_func               = _t.get("asso_func", "hmiou")
vial_count_cap          = _s.get("vial_count_cap", 10)
bg_gain                 = _p.get("bg_gain", 1.2)
bg_white_level          = _p.get("bg_white_level", 245)
bg_percentile           = _p.get("bg_percentile", 85.0)
bg_sample_stride        = _p.get("bg_sample_stride", 1)
default_end             = _p.get("default_end", 700)

# Auto-increment output directory: run_1, run_2, run_3, ...
_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
OUTPUT_PATH = str(_outputs_root / f"run_{_next_n}")

os.makedirs(OUTPUT_PATH, exist_ok=True)
PATH_TO_VID = RAW_VIDEO
print("Output dir:", OUTPUT_PATH)
print(f"asso_func={asso_func}, vial_count_cap={vial_count_cap}")
print(f"bg_gain={bg_gain}, bg_white_level={bg_white_level}, bg_percentile={bg_percentile}, bg_sample_stride={bg_sample_stride}, default_end={default_end}")

Output dir: ..\outputs\run_5
asso_func=hmiou, vial_count_cap=10
bg_gain=1.2, bg_white_level=245, bg_percentile=85.0, bg_sample_stride=1, default_end=700


## 2 — (Optional) Background subtraction

Opens a GUI: draw a crop ROI and choose a frame range.
The output is a `_pp.mp4` file with the **85th-percentile** background subtracted.
Skip this cell if your video already has good contrast.

In [4]:
preprocess = True  # set to True to run the GUI
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/001/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted.mp4"

if preprocess:
    PATH_TO_VID = Path(
        preprocess_bgsub_gui(
            video_path=RAW_VIDEO,
            out_mp4=None,
            default_end=default_end,
            gain=bg_gain,
            white_level=bg_white_level,
            bg_sample_stride=bg_sample_stride,
            bg_percentile=bg_percentile,
        )
    )
    print("Preprocessed video:", PATH_TO_VID)

Saved bgsub video: ..\2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m\13 DPE\001\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted_pp.mp4
Background (85.0th percentile) from 291 frames (stride=1).
Preprocessed video: ..\2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m\13 DPE\001\2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted_pp.mp4


## 3 — Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step — reuse the JSON for the same experimental setup.

In [5]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")

draw_and_save_vial_rois(
    video_path=str(PATH_TO_VID),
    roi_json_path=ROI_JSON,
)

Controls:
 - Drag mouse: add ROI
 - u: undo last ROI
 - r: reset all ROIs
 - q: finish (requires exactly 6 ROIs)
Added ROI 1 = (16, 4, 119, 405)
Added ROI 2 = (131, 3, 213, 407)
Added ROI 3 = (265, 3, 348, 408)
Added ROI 4 = (393, 0, 479, 407)
Added ROI 5 = (525, 1, 613, 409)
Added ROI 6 = (655, 4, 735, 408)
Saved ROIs to: ..\outputs\run_5\vial_rois.json


{'vial1': (16, 4, 119, 405),
 'vial2': (131, 3, 213, 407),
 'vial3': (265, 3, 348, 408),
 'vial4': (393, 0, 479, 407),
 'vial5': (525, 1, 613, 409),
 'vial6': (655, 4, 735, 408)}

## 4 — RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [6]:
WIDE_CSV = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")

df_wide = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    asso_func=asso_func,
    max_frames=None,
)

print(df_wide.shape)
df_wide.head()

WARNING  | Max age > max observations, increasing size of max observations...
SUCCESS  | OcSort: det_thresh=0.1, max_age=90, max_obs=50, min_hits=10, iou_threshold=0.01, per_class=False, asso_func=hmiou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


Saved: ..\outputs\run_5\tracks_wide_format.csv  (frames=291, tracks=136)
(291, 137)


,frame,id1,id2,id3,id4,id5,id6,id7,id8,id9,...,id281,id283,id286,id292,id301,id302,id313,id315,id317,id318
0,0,"(569.00, 350.97)","(724.28, 361.70)","(538.89, 350.31)","(78.11, 346.07)","(149.16, 301.62)","(695.37, 359.63)","(403.96, 322.14)","(411.88, 366.44)","(325.24, 381.60)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,"(566.73, 348.55)","(724.21, 361.66)","(538.80, 349.73)","(77.95, 346.08)","(151.02, 299.41)","(696.68, 359.17)","(403.97, 319.35)","(412.87, 362.99)","(325.21, 381.56)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,"(565.05, 346.84)","(724.26, 361.66)","(539.18, 348.71)","(78.01, 345.34)","(153.07, 297.64)","(696.77, 358.36)","(404.26, 316.43)","(414.74, 359.62)","(325.20, 381.63)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,"(563.42, 344.36)","(724.38, 361.25)","(540.23, 347.37)",NaN,"(155.41, 295.83)","(697.53, 356.94)","(404.31, 313.02)","(416.72, 356.50)","(325.19, 381.78)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,"(562.01, 341.31)","(724.21, 361.21)","(541.27, 346.15)",NaN,"(157.27, 295.34)","(697.56, 356.59)","(404.52, 310.09)","(418.84, 354.23)","(325.39, 381.50)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5 — Hungarian stitching

Links fragmented tracklets across gaps using motion-consistent assignment.
Output: long CSV with `orig_id` and `stitched_id` columns.

In [7]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")
LONG_CSV     = os.path.join(OUTPUT_PATH, "tracks_long_format.csv")

# Load vial ROIs
with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df   = wide_to_long(pd.read_csv(WIDE_CSV), out_csv=LONG_CSV)
tracklets = build_tracklets(long_df)

print(f"Built {len(tracklets)} tracklets from {long_df['orig_id'].nunique()} original IDs")

# Stitching — mode controlled by stitching_mode in config.yaml (default: per_vial)
stitched_df = stitch(
    long_df    = long_df,
    vial_rois  = vial_rois,
    tracklets  = tracklets,
    output_dir = OUTPUT_PATH,
)

stitched_df.to_csv(STITCHED_CSV, index=False)
print(f"\nSaved: {STITCHED_CSV}")
print(f"Stitched IDs: {stitched_df['stitched_id'].nunique()} (from {stitched_df['orig_id'].nunique()} original)")

Built 136 tracklets from 136 original IDs
  vial1 round 1: 31 -> 7 IDs (cap 10)
  vial2 round 1: 21 -> 10 IDs (cap 10)
  vial3 round 1: 26 -> 8 IDs (cap 10)
  vial4 round 1: 12 -> 7 IDs (cap 10)
  vial5 round 1: 17 -> 8 IDs (cap 10)
  vial6 round 1: 26 -> 9 IDs (cap 10)

Saved: ..\outputs\run_5\tracks_xy_stitched_long.csv
Stitched IDs: 52 (from 136 original)


## 6 — Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left → right within each vial).

In [8]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=_s.get("fps", 60),
)

print(df_compact.shape)
df_compact.head()

(6481, 8)


,frame,orig_id,x,y,stitched_id,vial_id,compact_id,fps
0,0,id1,569.00,350.97,id1,vial5,33,60.0
1,1,id1,566.73,348.55,id1,vial5,33,60.0
2,2,id1,565.05,346.84,id1,vial5,33,60.0
3,3,id1,563.42,344.36,id1,vial5,33,60.0
4,4,id1,562.01,341.31,id1,vial5,33,60.0


## 7 — Overlay video

Renders each fly as a coloured dot on the original video.

In [9]:
OVERLAY_MP4 = os.path.join(OUTPUT_PATH, "overlay_vials_shaded.mp4")

render_vial_overlay_video(
    video_path=str(PATH_TO_VID),
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
)

Video(OVERLAY_MP4, width=800)

Saved overlay video: ..\outputs\run_5\overlay_vials_shaded.mp4
